<h1 align="center">Proyecto Final: Calculadoras de Instrumentos de Deuda y Construcción de Curvas</h1>


Navarrete Ruvalcaba Diego  



<h2>Ejercicio 1: Calculadora de Precio de Cete</h2>

In [ ]:
"""
Ejercicio 1: Calculadora de Precio de CETE (México)

Modelo de CETE:
- Valor nominal fijo VN = 10.
- Tasa anual simple.
- Precio = VN / (1 + tasa_anual * fracción_de_año)
"""

from datetime import datetime, date, timedelta

# Usa los feriados oficiales de México.
try:
    import holidays
    MX_HOLIDAYS = holidays.Mexico()
except Exception:
    MX_HOLIDAYS = None

VN_CETE = 10.0  # Valor nominal del CETE

# 1. Fechas y días hábiles

def es_dia_habil(f):
    """
    Regresa True si la fecha f es día hábil en México.
    Día hábil = lunes a viernes y, si se tiene 'holidays', que no sea feriado.
    """
    if f.weekday() >= 5:  # 5 = sábado, 6 = domingo
        return False
    if MX_HOLIDAYS is not None and f in MX_HOLIDAYS:
        return False
    return True


def ajustar_a_siguiente_habil(f):
    """
    Si la fecha f no es hábil, la recorre día por día hasta el siguiente hábil.
    """
    while not es_dia_habil(f):
        f += timedelta(days=1)
    return f


def leer_fecha_ajustada(mensaje):
    """
    Lee una fecha por teclado con formato YYYY-MM-DD.
    - Si el formato es incorrecto, vuelve a pedir.
    - Si no es hábil, la ajusta al siguiente día hábil y lo indica.
    """
    while True:
        texto = input(mensaje).strip()
        try:
            f = datetime.strptime(texto, "%Y-%m-%d").date()
        except ValueError:
            print("Formato inválido. Usa YYYY-MM-DD (Ejemplo:2025-12-03).")
            continue
        f2 = ajustar_a_siguiente_habil(f)
        if f2 != f:
            print(f"La fecha {f} no es hábil. Se ajusta a {f2}.")
        return f2

# 2. Lectura robusta de números

def leer_float(mensaje, min_val=None, permitir_cero=True):
    """
    Lee un número flotante de forma robusta.
    - Acepta coma o punto como decimal.
    - Opcional: mínimo valor y prohibir cero.
    """
    while True:
        txt = input(mensaje).strip().replace(",", ".")
        try:
            val = float(txt)
        except ValueError:
            print("Valor inválido. Debe ser numérico (Ejemplo: 9.5).")
            continue
        if not permitir_cero and val == 0:
            print("El valor no puede ser cero.")
            continue
        if min_val is not None and val < min_val:
            print(f"El valor debe ser mayor o igual a {min_val}.")
            continue
        return val

# 3. Convenciones de día (“tipo de actual”)

def normalizar_daycount(texto):
    """
    Normaliza el tipo de 'actual' a uno de:
        'ACT/360', 'ACT/365', 'ACT/ACT', '30/360'
    Acepta variantes tipo: Actual/360, Actual365, Act/Act, Act30/360.
    """
    t = texto.strip().lower().replace(" ", "")
    if t in ("actual/360", "act/360", "actual360"):
        return "ACT/360"
    if t in ("actual/365", "act/365", "actual365"):
        return "ACT/365"
    if t in ("actual/actual", "actual/act", "act/act", "actact"):
        return "ACT/ACT"
    if t in ("actual30/360", "act30/360", "30/360", "act/30/360"):
        return "30/360"
    raise ValueError("Tipo de 'actual' inválido. Usa Actual/360, Actual/365, Act/Act o Act30/360.")


def fraccion_de_ano(f_val, f_venc, tipo_actual):
    """
    Calcula la fracción de año entre f_val y f_venc según la convención:

    - ACT/360 : días reales / 360
    - ACT/365 : días reales / 365
    - ACT/ACT : aproximación días reales / 365
    - 30/360  : convención 30/360
    """
    if f_venc <= f_val:
        raise ValueError("La fecha de vencimiento debe ser mayor a la de valuación.")

    dias = (f_venc - f_val).days
    t = tipo_actual.upper()

    if t == "ACT/360":
        return dias / 360.0
    if t == "ACT/365":
        return dias / 365.0
    if t == "ACT/ACT":
        # Aproximación sencilla: días reales entre 365
        return dias / 365.0
    if t == "30/360":
        d1, m1, a1 = f_val.day, f_val.month, f_val.year  # día, mes, año de valuación
        d2, m2, a2 = f_venc.day, f_venc.month, f_venc.year  # día, mes, año de vencimiento
        if d1 == 31:
            d1 = 30
        if d2 == 31 and d1 == 30:
            d2 = 30
        dias_30_360 = 360 * (a2 - a1) + 30 * (m2 - m1) + (d2 - d1)
        return dias_30_360 / 360.0

    raise ValueError("Tipo de 'actual' no reconocido internamente.")


# 4. Cálculo del precio del CETE

def calcular_precio_cete(f_val, f_venc, tipo_actual, tasa_anual):
    """
    Calcula el precio de un CETE (VN = 10) con tasa anual simple.

    Fórmula:
        Precio = VN_CETE / (1 + tasa_anual * fracción_de_año)
    """
    dias = (f_venc - f_val).days
    frac = fraccion_de_ano(f_val, f_venc, tipo_actual)
    denom = 1 + tasa_anual * frac
    if denom <= 0:
        raise ValueError("Denominador no válido (tasa * tiempo demasiado negativo).")
    precio = VN_CETE / denom
    return precio, dias, frac


# 5. Función principal

def main():
    print("\n---------------------------------------------")
    print(" EJERCICIO 1: CALCULADORA DE PRECIO DE CETE ")
    print("----------------------------------------------\n")


    try:
        # 1) Fechas (se ajustan a día hábil si es necesario)
        f_val = leer_fecha_ajustada("Ingresa la fecha de valuación (YYYY-MM-DD): ")
        f_venc = leer_fecha_ajustada("Ingresa la fecha de vencimiento (YYYY-MM-DD): ")

        # 2) Validar orden temporal
        if f_venc <= f_val:
            print("Error: la fecha de vencimiento debe ser mayor a la de valuación.")
            return

        # 3) Tipo de 'actual'
        tipo_txt = input("Tipo de 'actual' (Actual/360, Actual/365, Act/Act, Act30/360): ")
        tipo = normalizar_daycount(tipo_txt)

        # 4) Tasa anual simple en porcentaje
        tasa_pct = leer_float("Tasa anual simple del CETE (%): ", min_val=0.0, permitir_cero=False)
        tasa = tasa_pct / 100.0

        # 5) Cálculo de precio
        precio, dias, frac = calcular_precio_cete(f_val, f_venc, tipo, tasa)

        # 6) Resultados
        print("\n----------------- RESULTADOS -----------------")
        print(f"Fecha de valuación (ajustada):   {f_val}")
        print(f"Fecha de vencimiento (ajustada): {f_venc}")
        print(f"Días al vencimiento:             {dias}")
        print(f"Tipo de 'actual' usado:          {tipo}")
        print(f"Fracción de año:                 {frac:.8f}")
        print(f"Tasa anual usada:                {tasa*100:.4f}%")
        print(f"Valor nominal CETE:              {VN_CETE:.2f}")
        print("----------------------------------------------")
        print(f"Precio calculado del CETE:   {precio:.6f}")
        print("----------------------------------------------\n")

    except Exception as e:
        # Cualquier error inesperado se captura para que el programa no truene
        print("\n Ocurrió un error inesperado.")
        print("   Detalle técnico:", type(e).__name__, "-", e)


# Punto de entrada
if __name__ == "__main__":
    main()



---------------------------------------------
 EJERCICIO 1: CALCULADORA DE PRECIO DE CETE 
----------------------------------------------

Ingresa la fecha de valuación (YYYY-MM-DD): 2025-12-7
La fecha 2025-12-07 no es hábil. Se ajusta a 2025-12-08.
Ingresa la fecha de vencimiento (YYYY-MM-DD): 2026-12-08
Tipo de 'actual' (Actual/360, Actual/365, Act/Act, Act30/360): ACT ACT
Tasa anual simple del CETE (%): 8%
Valor inválido. Debe ser numérico (Ejemplo: 9.5).
Tasa anual simple del CETE (%): 8

----------------- RESULTADOS -----------------
Fecha de valuación (ajustada):   2025-12-08
Fecha de vencimiento (ajustada): 2026-12-08
Días al vencimiento:             365
Tipo de 'actual' usado:          ACT/ACT
Fracción de año:                 1.00000000
Tasa anual usada:                8.0000%
Valor nominal CETE:              10.00
----------------------------------------------
Precio calculado del CETE:   9.259259
----------------------------------------------



<h2>Ejercicio 2: Calculadora de Mbono</h2>

In [ ]:
# ==========================================
# Calculadora de precio de Mbono
# Precio sucio, precio limpio y duración
# ==========================================

from datetime import datetime, timedelta


# 1. Funciones auxiliares para validación de inputs

def pedir_fecha(mensaje):
    """
    Pide una fecha al usuario en formato AAAA-MM-DD.
    Valida que el formato sea correcto y la regresa como datetime.date.
    """
    while True:
        fecha_str = input(mensaje).strip()
        try:
            # Intentamos convertir la cadena a un objeto datetime
            fecha = datetime.strptime(fecha_str, "%Y-%m-%d").date()
            return fecha
        except ValueError:
            # Si falla el parseo, mostramos error y volvemos a pedir la fecha
            print("Error: la fecha debe ir en formato AAAA-MM-DD (ejemplo: 2025-12-04).")


def pedir_numero(mensaje):
    """
    Pide un número al usuario.
    Valida que realmente sea numérico (float) y regresa el valor.
    """
    while True:
        valor_str = input(mensaje).strip()
        try:
            valor = float(valor_str)
            return valor
        except ValueError:
            print("Error: sólo se admiten números. Vuelve a intentarlo.")


def pedir_convencion():
    """
    Pide el tipo de 'actual' / base de conteo de días.
    Acepta: actual/360, actual/365, actual/actual, 30/360 (sin importar mayúsculas).
    """
    opciones_validas = {"actual/360", "actual/365", "actual/actual", "30/360"}
    while True:
        conv = input("Introduce la convención de días "
                     "(Actual/360, Actual/365, Actual/Actual, 30/360): ").strip().lower()
        if conv in opciones_validas:
            return conv
        else:
            print("Error: convención no válida. Usa una de: Actual/360, Actual/365, Actual/Actual, 30/360.")


# 2. Funciones de negocio: días hábiles y year fraction

def es_dia_habil(fecha):
    """
    Verifica si una fecha es día hábil.
    Aquí suponemos que lunes-viernes son hábiles y sábado-domingo no lo son.
    (No se consideran feriados oficiales por simplicidad).
    """
    # weekday(): 0 = lunes, ..., 6 = domingo
    return fecha.weekday() < 5


def ajustar_a_siguiente_habil(fecha):
    """
    Si la fecha no es hábil, la recorre hacia adelante hasta el siguiente día hábil.
    """
    fecha_ajustada = fecha
    while not es_dia_habil(fecha_ajustada):
        fecha_ajustada += timedelta(days=1)
    return fecha_ajustada


def year_fraction(fecha_inicio, fecha_fin, conv):
    """
    Calcula la fracción de año entre dos fechas según la convención 'conv'.
    conv puede ser:
      - 'actual/360'
      - 'actual/365'
      - 'actual/actual'
      - '30/360'
    """
    if fecha_fin < fecha_inicio:
        raise ValueError("La fecha final es anterior a la fecha inicial en year_fraction.")

    if conv in ("actual/360", "actual/365", "actual/actual"):
        # Días naturales entre las dos fechas
        dias = (fecha_fin - fecha_inicio).days
        if conv == "actual/360":
            return dias / 360.0
        elif conv == "actual/365":
            return dias / 365.0
        else:  # actual/actual
            # Aproximación simple: usamos el año promedio de 365 días.
            # Para mayor precisión se podría ajustar por años bisiestos.
            return dias / 365.0
    else:
        # 30/360: versión sencilla para el conteo de días
        d1 = fecha_inicio.day
        m1 = fecha_inicio.month
        y1 = fecha_inicio.year

        d2 = fecha_fin.day
        m2 = fecha_fin.month
        y2 = fecha_fin.year

        if d1 == 31:
            d1 = 30
        if d2 == 31 and d1 == 30:
            d2 = 30

        dias_30_360 = 360 * (y2 - y1) + 30 * (m2 - m1) + (d2 - d1)
        return dias_30_360 / 360.0


# 3. Generación de flujo de cupones de un Mbono
# ---------------------------------------------------

def generar_fechas_cupones(fecha_valuacion, fecha_vencimiento):
    """
    Genera la lista de fechas de cupones (semi-anuales) desde la fecha
    de valuación hasta el vencimiento, excluyendo las fechas anteriores
    a la valuación.
    Asumimos que los cupones son cada 182 días (aprox 6 meses).
    """
    fechas = []
    # Empezamos desde el vencimiento y vamos hacia atrás
    fecha_actual = fecha_vencimiento
    while fecha_actual > fecha_valuacion:
        fechas.append(fecha_actual)
        fecha_actual -= timedelta(days=182)  # intervalo aproximado de cupón

    # Ordenamos de menor a mayor
    fechas.sort()
    return fechas


def obtener_ultima_y_siguiente_cupon(fecha_valuacion, fecha_vencimiento):
    """
    A partir de la valuación y el vencimiento, generamos los cupones
    y regresamos:
      - última fecha de cupón (<= valuación)
      - siguiente fecha de cupón (> valuación)
    Si la valuación cae exactamente en un cupón, la última fecha = valuación.
    """
    # Generamos cupones hacia atrás (como antes) pero incluyendo uno más previo
    fechas = []
    fecha_actual = fecha_vencimiento
    while True:
        fechas.append(fecha_actual)
        fecha_actual -= timedelta(days=182)
        if fecha_actual <= fecha_valuacion - timedelta(days=182 * 5):
            # corte de seguridad por si algo raro pasa con las fechas
            break

    fechas.sort()
    ultima = None
    siguiente = None

    for f in fechas:
        if f <= fecha_valuacion:
            ultima = f
        if f > fecha_valuacion:
            siguiente = f
            break

    # Si no encontramos 'ultima', la aproximamos como valuación menos 182 días
    if ultima is None:
        ultima = fecha_valuacion - timedelta(days=182)

    return ultima, siguiente


# -------------------------------------------------
# 4. Cálculo de precio y duración del Mbono
# -------------------------------------------------

def valorar_mbono(fecha_val, fecha_vto, conv, cupon_anual, ytm_anual, valor_nominal=100.0):
    """
    Calcula precio sucio, limpio y duración (Macaulay y modificada) de un Mbono.

    Parámetros:
      - fecha_val: fecha de valuación (date)
      - fecha_vto: fecha de vencimiento (date)
      - conv: convención de días (str)
      - cupon_anual: tasa cupón anual en formato decimal (ej. 0.08 = 8%)
      - ytm_anual: rendimiento anual (YTM) en decimal
      - valor_nominal: valor nominal del bono (por defecto 100)

    Retorna:
      (precio_sucio, precio_limpio, dur_macaulay, dur_modificada)
    """
    # Pago de cupón semestral
    cupon_semestral = valor_nominal * cupon_anual / 2.0

    # Generamos fechas de cupones futuras
    fechas_cupones = generar_fechas_cupones(fecha_val, fecha_vto)

    # Cálculo del precio sucio y de la duración (Macaulay)
    precio_sucio = 0.0
    suma_t_pv = 0.0  # para duración

    for fecha_pago in fechas_cupones:
        # Tiempo en años usando la convención seleccionada
        t = year_fraction(fecha_val, fecha_pago, conv)

        # Flujo de caja: cupón y, en el último pago, el principal
        flujo = cupon_semestral
        if fecha_pago == fecha_vto:
            flujo += valor_nominal

        # Descuento usando YTM con capitalización semestral
        df = 1.0 / (1.0 + ytm_anual / 2.0) ** (2.0 * t)

        pv = flujo * df
        precio_sucio += pv

        # Acumulamos para la duración de Macaulay
        suma_t_pv += t * pv

    # Duración de Macaulay en años
    dur_macaulay = suma_t_pv / precio_sucio if precio_sucio != 0 else 0.0

    # Duración modificada
    dur_modificada = dur_macaulay / (1.0 + ytm_anual / 2.0)

    # Cálculo del interés devengado (accrued interest) para el precio limpio
    ultima_cupon, siguiente_cupon = obtener_ultima_y_siguiente_cupon(fecha_val, fecha_vto)

    # Fracción de año desde el último cupón hasta la fecha de valuación
    fraccion_desde_ultimo = year_fraction(ultima_cupon, fecha_val, conv)

    # Fracción de año entre el último y el siguiente cupón (para saber qué parte del cupón ya se devengó)
    if siguiente_cupon is not None:
        fraccion_periodo = year_fraction(ultima_cupon, siguiente_cupon, conv)
    else:
        # Si no hay siguiente cupón (en vencimiento), tomamos 0.5 años (aprox)
        fraccion_periodo = 0.5

    proporcion = fraccion_desde_ultimo / fraccion_periodo if fraccion_periodo != 0 else 0.0
    interes_devengado = cupon_semestral * proporcion

    precio_limpio = precio_sucio - interes_devengado

    return precio_sucio, precio_limpio, dur_macaulay, dur_modificada


# 5. Programa principal

def main():
    print("=== Calculadora de Mbono (precio sucio, limpio y duración) ===")
    print("Formato de fechas: AAAA-MM-DD (ejemplo: 2025-12-04)")
    print("Las tasas deben ingresarse en porcentaje (ejemplo: 8 para 8%).")
    print()

    # 1) Pedimos fechas con validación de formato
    fecha_valuacion = pedir_fecha("Fecha de valuación: ")
    fecha_vencimiento = pedir_fecha("Fecha de vencimiento: ")

    # 2) Verificamos que vencimiento > valuación
    if fecha_vencimiento <= fecha_valuacion:
        print("Error: la fecha de vencimiento debe ser mayor a la fecha de valuación.")
        return

    # 3) Ajuste de días no hábiles
    if not es_dia_habil(fecha_valuacion):
        print("Advertencia: la fecha de valuación no es día hábil. "
              "Se ajustará al siguiente día hábil.")
        fecha_valuacion = ajustar_a_siguiente_habil(fecha_valuacion)

    if not es_dia_habil(fecha_vencimiento):
        print("Advertencia: la fecha de vencimiento no es día hábil. "
              "Se ajustará al siguiente día hábil.")
        fecha_vencimiento = ajustar_a_siguiente_habil(fecha_vencimiento)

    # 4) Pedimos convención de días
    conv = pedir_convencion()

    # 5) Pedimos tasas y validamos que sean numéricas
    cupon_pct = pedir_numero("Tasa cupón anual (%): ")
    ytm_pct = pedir_numero("YTM anual (%): ")

    # Validación básica: no permitir tasas negativas 4
    if cupon_pct < 0 or ytm_pct < 0:
        print("Error: las tasas no pueden ser negativas.")
        return

    # Convertimos de porcentaje a decimal
    cupon_anual = cupon_pct / 100.0
    ytm_anual = ytm_pct / 100.0

    # 6) Llamamos a la función de valoración
    try:
        precio_sucio, precio_limpio, dur_macaulay, dur_modificada = valorar_mbono(
            fecha_valuacion,
            fecha_vencimiento,
            conv,
            cupon_anual,
            ytm_anual,
            valor_nominal=100.0  # Valor nominal típico de Mbono
        )
    except ValueError as e:
        print(f"Error durante el cálculo: {e}")
        return

    # 7) Mostramos resultados
    print("\n--- Resultados ---")
    print(f"Fecha de valuación ajustada  : {fecha_valuacion}")
    print(f"Fecha de vencimiento ajustada: {fecha_vencimiento}")
    print(f"Precio sucio  : {precio_sucio:,.4f}")
    print(f"Precio limpio : {precio_limpio:,.4f}")
    print(f"Duración Macaulay  (años): {dur_macaulay:,.4f}")
    print(f"Duración modificada (años): {dur_modificada:,.4f}")


# Punto de entrada del programa
if __name__ == "__main__":
    main()


=== Calculadora de Mbono (precio sucio, limpio y duración) ===
Formato de fechas: AAAA-MM-DD (ejemplo: 2025-12-04)
Las tasas deben ingresarse en porcentaje (ejemplo: 8 para 8%).

Fecha de valuación: 2023-03-12
Fecha de vencimiento: 2025-12-20
Advertencia: la fecha de valuación no es día hábil. Se ajustará al siguiente día hábil.
Advertencia: la fecha de vencimiento no es día hábil. Se ajustará al siguiente día hábil.
Introduce la convención de días (Actual/360, Actual/365, Actual/Actual, 30/360): Actual/360
Tasa cupón anual (%): .13
YTM anual (%): 4

--- Resultados ---
Fecha de valuación ajustada  : 2023-03-13
Fecha de vencimiento ajustada: 2025-12-22
Precio sucio  : 89.8013
Precio limpio : 89.7738
Duración Macaulay  (años): 2.8142
Duración modificada (años): 2.7590


<h2>Ejercicio 3: Calculadora de Udibono</h2>


In [ ]:
# ==========================================
# Calculadora de precio de Udibonos
# Precio sucio, precio limpio y duración
# ==========================================

from datetime import datetime, timedelta, date

# Intentamos usar feriados oficiales de México
try:
    import holidays
    MX_HOLIDAYS = holidays.Mexico()
except Exception:
    MX_HOLIDAYS = None

# Valor nominal del Udibono en UDIS
VN_UDI = 100.0


# 1. Funciones auxiliares para validación de inputs
#Para definir estas funciones cada una usa un ciclo que garantiza la validez de los datos.

def pedir_fecha(mensaje):
    """
    Pide una fecha al usuario en formato AAAA-MM-DD.
    Valida que el formato sea correcto y la regresa como datetime.date.
    """
    while True:
        fecha_str = input(mensaje).strip()
        try:
            fecha = datetime.strptime(fecha_str, "%Y-%m-%d").date()
            return fecha
        except ValueError:
            print("Error: la fecha debe ir en formato AAAA-MM-DD (ejemplo: 2025-12-04).")


def pedir_numero(mensaje):
    """
    Pide un número al usuario.
    Valida que realmente sea numérico (float) y regresa el valor.
    """
    while True:
        valor_str = input(mensaje).strip()
        try:
            valor = float(valor_str)
            return valor
        except ValueError:
            print("Error: sólo se admiten números. Vuelve a intentarlo.")


def pedir_convencion():
    """
    Pide el tipo de 'actual' / base de conteo de días.
    Acepta: actual/360, actual/365, actual/actual, 30/360 (sin importar mayúsculas).
    """
    opciones_validas = {"actual/360", "actual/365", "actual/actual", "30/360"}
    while True:
        conv = input("Introduce la convención de días "
                     "(Actual/360, Actual/365, Actual/Actual, 30/360): ").strip().lower()
        if conv in opciones_validas:
            return conv
        else:
            print("Error: convención no válida. Usa una de: Actual/360, Actual/365, Actual/Actual, 30/360.")


# 2. Funciones de negocio: días hábiles y year fraction

def es_dia_habil(fecha: date) -> bool:
    """
    Verifica si una fecha es día hábil.
    - Lunes a viernes son hábiles.
    - Sábado y domingo no.
    - Si MX_HOLIDAYS está disponible, también excluye feriados oficiales de México.
    """
    if fecha.weekday() >= 5:  # 5 = sábado, 6 = domingo
        return False
    if MX_HOLIDAYS is not None and fecha in MX_HOLIDAYS:
        return False
    return True


def ajustar_a_siguiente_habil(fecha: date) -> date:
    """
    Si la fecha no es hábil, la recorre hacia adelante hasta el siguiente día hábil.
    """
    fecha_ajustada = fecha
    while not es_dia_habil(fecha_ajustada):
        fecha_ajustada += timedelta(days=1)
    return fecha_ajustada


def year_fraction(fecha_inicio, fecha_fin, conv):
    """
    Calcula la fracción de año entre dos fechas según la convención 'conv'.
    conv puede ser:
      - 'actual/360'
      - 'actual/365'
      - 'actual/actual'
      - '30/360'
    """
    if fecha_fin < fecha_inicio:
        raise ValueError("La fecha final es anterior a la fecha inicial en year_fraction.")

    if conv in ("actual/360", "actual/365", "actual/actual"):
        # Días naturales entre las dos fechas
        dias = (fecha_fin - fecha_inicio).days
        if conv == "actual/360":
            return dias / 360.0
        elif conv == "actual/365":
            return dias / 365.0
        else:  # actual/actual
            # Aproximación simple: usamos 365 días.
            return dias / 365.0
    else:
        # 30/360: versión sencilla (USA) para el conteo de días
        d1 = fecha_inicio.day
        m1 = fecha_inicio.month
        y1 = fecha_inicio.year

        d2 = fecha_fin.day
        m2 = fecha_fin.month
        y2 = fecha_fin.year

        if d1 == 31:
            d1 = 30
        if d2 == 31 and d1 == 30:
            d2 = 30

        dias_30_360 = 360 * (y2 - y1) + 30 * (m2 - m1) + (d2 - d1)
        return dias_30_360 / 360.0


# 3. Generación de flujo de cupones de un Udibono


def generar_fechas_cupones(fecha_valuacion, fecha_vencimiento):
    """
    Genera la lista de fechas de cupones (semi-anuales) desde la fecha
    de valuación hasta el vencimiento, excluyendo las fechas anteriores
    a la valuación.
    Asumimos que los cupones son cada 182 días (aprox 6 meses).
    """
    fechas = []
    fecha_actual = fecha_vencimiento
    while fecha_actual > fecha_valuacion:
        fechas.append(fecha_actual)
        fecha_actual -= timedelta(days=182)  # intervalo aproximado de cupón

    fechas.sort()
    return fechas


def obtener_ultima_y_siguiente_cupon(fecha_valuacion, fecha_vencimiento):
    """
    A partir de la valuación y el vencimiento, generamos los cupones
    y regresamos:
      - última fecha de cupón (<= valuación)
      - siguiente fecha de cupón (> valuación)
    Si la valuación cae exactamente en un cupón, la última fecha = valuación.
    """
    fechas = []
    fecha_actual = fecha_vencimiento
    while True:
        fechas.append(fecha_actual)
        fecha_actual -= timedelta(days=182)
        if fecha_actual <= fecha_valuacion - timedelta(days=182 * 5):
            # corte de seguridad
            break

    fechas.sort()
    ultima = None
    siguiente = None

    for f in fechas:
        if f <= fecha_valuacion:
            ultima = f
        if f > fecha_valuacion:
            siguiente = f
            break

    if ultima is None:
        ultima = fecha_valuacion - timedelta(days=182)

    return ultima, siguiente


# 4. Cálculo de precio y duración del Udibono

def valorar_udibono(fecha_val, fecha_vto, conv, cupon_anual, ytm_anual, valor_udi, valor_nominal_udi=VN_UDI):
    """
    Calcula precio sucio, limpio y duración (Macaulay y modificada) de un Udibono en pesos.

    Parámetros:
      - fecha_val: fecha de valuación (date)
      - fecha_vto: fecha de vencimiento (date)
      - conv: convención de días (str)
      - cupon_anual: tasa cupón anual real en UDIS (decimal, ej. 0.04 = 4%)
      - ytm_anual: rendimiento real anual (YTM) en decimal
      - valor_udi: valor actual de la UDI en pesos
      - valor_nominal_udi: valor nominal del Udibono en UDIS (por defecto 100)

    Retorna:
      (precio_sucio_pesos, precio_limpio_pesos, dur_macaulay, dur_modificada)
    """
    # Cupón semestral en UDIS
    cupon_semestral_udi = valor_nominal_udi * cupon_anual / 2.0

    # Convertimos a pesos con el valor de la UDI
    cupon_semestral_pesos = cupon_semestral_udi * valor_udi
    principal_pesos = valor_nominal_udi * valor_udi

    # Generamos fechas de cupones futuras
    fechas_cupones = generar_fechas_cupones(fecha_val, fecha_vto)

    # Cálculo del precio sucio y de la duración (Macaulay)
    precio_sucio = 0.0
    suma_t_pv = 0.0  # para duración

    for fecha_pago in fechas_cupones:
        # Tiempo en años usando la convención de días
        t = year_fraction(fecha_val, fecha_pago, conv)

        # Flujo de caja en pesos: cupón, y en el último pago también el principal indexado
        flujo = cupon_semestral_pesos
        if fecha_pago == fecha_vto:
            flujo += principal_pesos

        # Descuento usando YTM real con capitalización semestral
        df = 1.0 / (1.0 + ytm_anual / 2.0) ** (2.0 * t)

        pv = flujo * df
        precio_sucio += pv

        # Para la duración de Macaulay
        suma_t_pv += t * pv

    # Duración de Macaulay en años
    dur_macaulay = suma_t_pv / precio_sucio if precio_sucio != 0 else 0.0

    # Duración modificada
    dur_modificada = dur_macaulay / (1.0 + ytm_anual / 2.0)

    # Cálculo del interés devengado (accrued interest) para el precio limpio
    ultima_cupon, siguiente_cupon = obtener_ultima_y_siguiente_cupon(fecha_val, fecha_vto)

    # Fracción de año desde el último cupón hasta la fecha de valuación
    fraccion_desde_ultimo = year_fraction(ultima_cupon, fecha_val, conv)

    # Fracción de año entre el último y el siguiente cupón
    if siguiente_cupon is not None:
        fraccion_periodo = year_fraction(ultima_cupon, siguiente_cupon, conv)
    else:
        # Si no hay siguiente cupón (en vencimiento), tomamos 0.5 años aprox
        fraccion_periodo = 0.5

    proporcion = fraccion_desde_ultimo / fraccion_periodo if fraccion_periodo != 0 else 0.0
    interes_devengado = cupon_semestral_pesos * proporcion

    precio_limpio = precio_sucio - interes_devengado

    return precio_sucio, precio_limpio, dur_macaulay, dur_modificada


# 5. Programa principal

def main():
    print("=== Calculadora de Udibonos (precio sucio, limpio y duración) ===")
    print("Formato de fechas: AAAA-MM-DD (ejemplo: 2025-12-04)")
    print("Las tasas deben ingresarse en porcentaje (ejemplo: 4 para 4%).")
    print()

    # 1) Fechas
    fecha_valuacion = pedir_fecha("Fecha de valuación: ")
    fecha_vencimiento = pedir_fecha("Fecha de vencimiento: ")

    # 2) Validamos que vencimiento > valuación
    if fecha_vencimiento <= fecha_valuacion:
        print("Error: la fecha de vencimiento debe ser mayor a la fecha de valuación.")
        return

    # 3) Ajuste de días no hábiles (usando fines de semana y, si se puede, feriados MX)
    if not es_dia_habil(fecha_valuacion):
        print("Advertencia: la fecha de valuación no es día hábil. "
              "Se ajustará al siguiente día hábil.")
        fecha_valuacion = ajustar_a_siguiente_habil(fecha_valuacion)

    if not es_dia_habil(fecha_vencimiento):
        print("Advertencia: la fecha de vencimiento no es día hábil. "
              "Se ajustará al siguiente día hábil.")
        fecha_vencimiento = ajustar_a_siguiente_habil(fecha_vencimiento)

    # 4) Convención de días
    conv = pedir_convencion()

    # 5) Tasas y valor UDI
    cupon_pct = pedir_numero("Tasa cupón anual real en UDIS (%): ")
    valor_udi = pedir_numero("Valor actual de la UDI (en pesos): ")
    ytm_pct = pedir_numero("YTM real anual (%): ")

    # Validación básica
    if cupon_pct < 0 or ytm_pct < 0 or valor_udi <= 0:
        print("Error: las tasas no pueden ser negativas y el valor de la UDI debe ser positivo.")
        return

    # Convertimos a decimal
    cupon_anual = cupon_pct / 100.0
    ytm_anual = ytm_pct / 100.0

    # 6) Valoración
    try:
        precio_sucio, precio_limpio, dur_macaulay, dur_modificada = valorar_udibono(
            fecha_valuacion,
            fecha_vencimiento,
            conv,
            cupon_anual,
            ytm_anual,
            valor_udi,
            valor_nominal_udi=VN_UDI
        )
    except ValueError as e:
        print(f"Error durante el cálculo: {e}")
        return

    # 7) Resultados
    print("\n--- Resultados ---")
    print(f"Fecha de valuación ajustada  : {fecha_valuacion}")
    print(f"Fecha de vencimiento ajustada: {fecha_vencimiento}")
    print(f"Valor nominal en UDIS        : {VN_UDI:,.2f}")
    print(f"Valor UDI (pesos)            : {valor_udi:,.6f}")
    print(f"Precio sucio (pesos)         : {precio_sucio:,.4f}")
    print(f"Precio limpio (pesos)        : {precio_limpio:,.4f}")
    print(f"Duración Macaulay  (años)    : {dur_macaulay:,.4f}")
    print(f"Duración modificada (años)   : {dur_modificada:,.4f}")


if __name__ == "__main__":
    main()


=== Calculadora de Udibonos (precio sucio, limpio y duración) ===
Formato de fechas: AAAA-MM-DD (ejemplo: 2025-12-04)
Las tasas deben ingresarse en porcentaje (ejemplo: 4 para 4%).

Fecha de valuación: 2025-03015
Error: la fecha debe ir en formato AAAA-MM-DD (ejemplo: 2025-12-04).
Fecha de valuación: 2025-03-15
Fecha de vencimiento: 2030-12-15
Advertencia: la fecha de valuación no es día hábil. Se ajustará al siguiente día hábil.
Advertencia: la fecha de vencimiento no es día hábil. Se ajustará al siguiente día hábil.
Introduce la convención de días (Actual/360, Actual/365, Actual/Actual, 30/360): actual/365
Tasa cupón anual real en UDIS (%): 4
Valor actual de la UDI (en pesos): 7.95
YTM real anual (%): 4.5

--- Resultados ---
Fecha de valuación ajustada  : 2025-03-18
Fecha de vencimiento ajustada: 2030-12-16
Valor nominal en UDIS        : 100.00
Valor UDI (pesos)            : 7.950000
Precio sucio (pesos)         : 782.8804
Precio limpio (pesos)        : 775.4545
Duración Macaulay  (a

<h2>Ejercicio 4: Curva Cupón Cero y Curvas Forward</h2>

In [ ]:
"""Restricciones que cubre el codigo:
- Plazos automáticos en semestres
- Tasas (0,100], decimal <1, porcentaje >=1
- Acepta comas como puntos
- No acepta letras
- No acepta símbolo %
- Conversión decimal→porcentaje mostrada al usuario
- Repetir el ingreso de otra curva o salir
"""

import math

# UTILIDAD PARA LIMPIAR INPUT (comas→puntos)

def limpiar_numero(txt):
    """Convierte comas a puntos y elimina espacios."""
    return txt.replace(",", ".").strip()


# VALIDACIONES
#Valores que permite y no permite el codigo, tanto para el numero de bonos, como para las tasas.
#Sirve para evitar que el codigo se "rompa" y le permita al usuario volver a ingresar
#valores sin que tenga que repetir todo el procesos de ingreso de datos
def pedir_entero_positivo(mensaje):
    """Pide un entero positivo sin tronar."""
    while True:
        try:
            x = input(mensaje)
            x = limpiar_numero(x)

            if "%" in x:
                print("No coloques el signo de porcentaje (%). Intenta de nuevo.")
                continue

            if not x.replace(".", "", 1).isdigit():
                print("Solo números enteros positivos. Intenta de nuevo.")
                continue

            n = float(x)

            if n <= 0:
                print("Debe ser un número positivo. Intenta de nuevo.")
                continue

            if abs(n - int(n)) > 1e-9:
                print("Debe ser un número ENTERO (ej. 3, 4, 5). Intenta de nuevo.")
                continue

            return int(n)

        except:
            print("Entrada inválida. Intenta de nuevo.")


def pedir_tasa(mensaje):
    """
    Reglas:
      - Solo se admiten tasas en (0,100]
      - Acepta comas
      - No acepta letras
      - No acepta %
      - Decimal <1, porcentaje >=1
      - Muestra conversión cuando sea necesario
    """
    while True:
        txt = input(mensaje).strip()

        # reemplazar coma por punto
        txt = limpiar_numero(txt)

        # prohibir %
        if "%" in txt:
            print("No coloques el signo de porcentaje (%). Intenta de nuevo.")
            continue

        # validar que NO tenga letras
        if not txt.replace(".", "", 1).isdigit():
            print("Las tasas no deben contener letras ni símbolos. Intenta de nuevo.")
            continue

        try:
            t = float(txt)

            if t <= 0 or t > 100:
                print("La tasa debe estar en el rango (0,100]. Intenta de nuevo.")
                continue

            # decimal
            if t < 1:
                print(f"   ↳ Tasa decimal ({t}), equivalente a {t*100:.4f}%")
                return t

            # porcentaje
            print(f"   ↳ Interpretando {t}% como porcentaje.")
            return t / 100.0

        except:
            print(" Entrada inválida. Intenta de nuevo.")


# FLUJOS, PRECIOS Y BOOTSTRAP


def generar_tiempos_flujo(plazo):
    return [0.5 * k for k in range(1, int(plazo*2) + 1)]


def precio_sucio(cupon_dec, ytm_dec, plazo, nominal=100):
    C = cupon_dec * nominal
    tiempos = generar_tiempos_flujo(plazo)
    P = 0.0
#Aquí seguimos el método del excel que vimos en la ayudantía precio sucio=precio con las nuevas tasas, construcción recursiva
    # cupones intermedios
    for t in tiempos[:-1]:
        P += C / ((1 + ytm_dec)**t)

    # final
    P += (C + nominal) / ((1 + ytm_dec)**tiempos[-1])
    return P

#funciona como el despeje

def bootstrap_curve(plazos, cupones_dec, precios, nominal=100):
    curva = {}
    for idx, t in enumerate(plazos):
        cupon = cupones_dec[idx]
        C = cupon * nominal
        Pk = precios[idx]

        pv_prev = 0.0
        for t_prev in plazos[:idx]:
            pv_prev += C / ((1 + curva[t_prev])**t_prev)

        rem = Pk - pv_prev
        if rem <= 0:
            print(f" Advertencia: precio inconsistente en t={t}. Ajustando…")
            rem = abs(rem) + 1e-9
#por si sale algo negativo a la hora de despejar
        base = (C + nominal) / rem
        curva[t] = base**(1/t) - 1

    return curva


# FORWARDS

def forward_rate(z1, t1, z2, t2):
    return ((1+z2)**t2 / (1+z1)**t1)**(1/(t2-t1)) - 1


def forward_1y3y_extendido(curva):
    """Forward 1→4 usando tu regla: z4=z3."""
    #aquí repetimos z3 para z4
    if 1.0 not in curva or 3.0 not in curva:
        return None
    z1 = curva[1.0]
    z3 = curva[3.0]
    z4 = z3
    return ((1+z4)**4 / (1+z1))**(1/3) - 1



# CURVA COMPLETA


def ejecutar_curva():
    print("\n=== CONSTRUCCIÓN DE CURVA CUPÓN-CERO ===\n")

    nominal = 100.0

    # número de bonos
    n = pedir_entero_positivo("¿Cuántos bonos vas a ingresar?: ")

    # plazos automáticos, para que el usuario ni tenga que ponerlos uno por uno
    plazos = [0.5 * (i+1) for i in range(n)]
    print("\nPlazos generados automáticamente:")
    print("  " + ", ".join(str(t) for t in plazos))

    cupones_dec = [ ]
    ytms_dec = []
    precios = []
#aquí voy guardando los precios sucios y las z's ya calculadas para usarlas recursivamente
    for t in plazos:
        print(f"\n--- Bono con plazo {t} años ---")

        cup = pedir_tasa("Tasa cupón .: ")
        ytm = pedir_tasa("YTM anual: ")

        cupones_dec.append(cup)
        ytms_dec.append(ytm)

        P = precio_sucio(cup, ytm, t, nominal)
        precios.append(P)

        print(f"Precio sucio para t={t}: {P:.6f}")

    print("\n=== CURVA CUPÓN-CERO ===")
    curva = bootstrap_curve(plazos, cupones_dec, precios, nominal)
    for t in plazos:
        print(f"z({t}) = {curva[t]*100:.4f}%")

    print("\n=== FORWARDS ===")
    if 1.0 in curva and 2.0 in curva:
        f = forward_rate(curva[1.0], 1.0, curva[2.0], 2.0)
        print(f"1y1y (1→2): {f*100:.4f}%")

    if 1.0 in curva and 3.0 in curva:
        f = forward_rate(curva[1.0], 1.0, curva[3.0], 3.0)
        print(f"1y2y (1→3): {f*100:.4f}%")

    f_ext = forward_1y3y_extendido(curva)
    if f_ext is not None:
        print(f"1y3y extendido (1→4, usando z4=z3): {f_ext*100:.4f}%")

    print("\n=== FIN DE CURVA ===\n")


# MENÚ FINAL para facilitar el uso de la calculadora

def main():
    while True:
        ejecutar_curva()

        print("¿Qué deseas ahora?")
        print("  1) Calcular otra curva")
        print("  2) Salir")

        op = input("Elige 1 o 2: ").strip()

        if op == "1":
            continue
        elif op == "2":
            print("\nSaliendo... ¡Bye!\n")
            break
        else:
            print("Opción no válida, repitiendo menú.")


if __name__ == "__main__":
    main()


=== CONSTRUCCIÓN DE CURVA CUPÓN-CERO ===

¿Cuántos bonos vas a ingresar?: 6

Plazos generados automáticamente:
  0.5, 1.0, 1.5, 2.0, 2.5, 3.0

--- Bono con plazo 0.5 años ---
Tasa cupón .: 6.8
   ↳ Interpretando 6.8% como porcentaje.
YTM anual: 0.0675
   ↳ Tasa decimal (0.0675), equivalente a 6.7500%
Precio sucio para t=0.5: 103.368285

--- Bono con plazo 1.0 años ---
Tasa cupón .: 7
   ↳ Interpretando 7.0% como porcentaje.
YTM anual: 6,9
   ↳ Interpretando 6.9% como porcentaje.
Precio sucio para t=1.0: 106.863865

--- Bono con plazo 1.5 años ---
Tasa cupón .: 7.2
   ↳ Interpretando 7.2% como porcentaje.
YTM anual: 7.05
   ↳ Interpretando 7.05% como porcentaje.
Precio sucio para t=1.5: 110.471204

--- Bono con plazo 2.0 años ---
Tasa cupón .: 7.4
   ↳ Interpretando 7.4% como porcentaje.
YTM anual: 7,2
   ↳ Interpretando 7.2% como porcentaje.
Precio sucio para t=2.0: 114.174917

--- Bono con plazo 2.5 años ---
Tasa cupón .: 7,55
   ↳ Interpretando 7.55% como porcentaje.
YTM anual: 7,30